In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamp columns to UTC datetimes (per rule)
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) metric.csv: unique cmdb_id (sorted) and kpi_name values and counts (compact)
metric_cmdbs = pd.DataFrame({'cmdb_id': sorted(metric_df['cmdb_id'].dropna().unique())})
metric_kpi_names = pd.DataFrame({'kpi_name': sorted(metric_df['kpi_name'].dropna().unique())})
metric_kpi_counts_top50 = (
    metric_df.groupby('kpi_name').size().reset_index(name='count')
    .sort_values(by='count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# 2) trace.csv: unique cmdb_id (sorted) and trace_name values and counts
trace_cmdbs = pd.DataFrame({'cmdb_id': sorted(trace_df['cmdb_id'].dropna().unique())})
trace_names = pd.DataFrame({'trace_name': sorted(trace_df['trace_name'].dropna().unique())})
trace_name_counts = (
    trace_df.groupby('trace_name').size().reset_index(name='count')
    .sort_values(by='count', ascending=False)
    .reset_index(drop=True)
)

# 3) log.csv: unique cmdb_id (sorted) and log_name values and counts
log_cmdbs = pd.DataFrame({'cmdb_id': sorted(log_df['cmdb_id'].dropna().unique())})
log_names = pd.DataFrame({'log_name': sorted(log_df['log_name'].dropna().unique())})
log_name_counts = (
    log_df.groupby('log_name').size().reset_index(name='count')
    .sort_values(by='count', ascending=False)
    .reset_index(drop=True)
)

# 4) error_logs.csv: sample up to first 20 rows (all columns)
error_logs_sample = error_logs_df.head(20)

# Display the compact summaries (only requested aggregates and sample)
metric_cmdbs, metric_kpi_names, metric_kpi_counts_top50, trace_cmdbs, trace_names, trace_name_counts.head(50), log_cmdbs, log_names, log_name_counts.head(50), error_logs_sample

```
Out[1]:
```
# Prepare a concise plain-English summary based on previously loaded results
metric_cmdb_count = metric_cmdbs.shape[0]
metric_kpis = metric_kpi_names['kpi_name'].tolist()
metric_top_counts = metric_kpi_counts_top50.head(8)

trace_cmdb_count = trace_cmdbs.shape[0]
trace_names_count = len(trace_names)
trace_top = trace_name_counts.head(5)

log_cmdb_count = log_cmdbs.shape[0]
log_kpis = log_names['log_name'].tolist()
log_counts = log_name_counts

error_sample_rows = error_logs_sample.shape[0]

summary = (
    f"Summary:\n\n"
    f"1) metric.csv: {metric_cmdb_count} unique cmdb_id values (sorted).\n"
    f"   KPI names ({len(metric_kpis)}): {', '.join(metric_kpis)}.\n"
    f"   Top KPI counts (top 8): " +
    ", ".join([f"{row.kpi_name} {int(row['count'])}" for _, row in metric_top_counts.iterrows()]) + ".\n\n"
    f"2) trace.csv: {trace_cmdb_count} unique cmdb_id values (sorted).\n"
    f"   {trace_names_count} unique trace_name values.\n"
    f"   Top 5 trace_name counts: " +
    ", ".join([f"{row.trace_name} {int(row['count'])}" for _, row in trace_top.iterrows()]) + ".\n\n"
    f"3) log.csv: {log_cmdb_count} unique cmdb_id values (sorted).\n"
    f"   Log names: {', '.join(log_kpis)}.\n"
    f"   Counts: " + ", ".join([f"{r.log_name} {int(r['count'])}" for _, r in log_counts.iterrows()]) + ".\n\n"
    f"4) error_logs.csv: sample rows available = {error_sample_rows} (empty sample shown).\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id
0    ts-admin-basic-info-service
1         ts-admin-order-service
2         ts-admin-route-service
3        ts-admin-travel-service
4          ts-admin-user-service
..                           ...
63                 ts-user-mongo
64               ts-user-service
65  ts-verification-code-service
66              ts-voucher-mysql
67            ts-voucher-service

[68 rows x 1 columns],      kpi_name
0         cpu
1      diskio
2       error
3  latency-50
4  latency-90
5         mem
6      socket
7    workload,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4    workload    700
5  latency-50    697
6  latency-90    697
7       error    125,                         cmdb_id
0                          root
1   ts-admin-basic-info-service
2       ts-admin-travel-service
3          ts-assurance-service
4               ts-auth-service
..                          ...
23        ts-ticketinfo-service
24             ts-train-service
25            ts-travel-service
26           ts-travel2-service
27              ts-user-service

[28 rows x 1 columns],                                             trace_name
0                        trace.from_root.duration_mean
1                         trace.from_root.duration_p95
2                           trace.from_root.error_rate
3                            trace.from_root.row_count
4    trace.from_ts-admin-basic-info-service.duratio...
..                                                 ...
215              trace.to_ts-travel2-service.row_count
216             trace.to_ts-user-service.duration_mean
217              trace.to_ts-user-service.duration_p95
218                trace.to_ts-user-service.error_rate
219                 trace.to_ts-user-service.row_count

[220 rows x 1 columns],                                         trace_name  count
0                    trace.from_root.duration_mean    345
1                     trace.from_root.duration_p95    345
2                       trace.from_root.error_rate    345
3                        trace.from_root.row_count    345
4   trace.from_ts-preserve-other-service.row_count    257
..                                             ...    ...
45          trace.to_ts-travel2-service.error_rate    123
46        trace.to_ts-travel2-service.duration_p95    123
47       trace.to_ts-travel2-service.duration_mean    123
48              trace.to_ts-seat-service.row_count    121
49             trace.to_ts-seat-service.error_rate    121

[50 rows x 2 columns],                         cmdb_id
0   ts-admin-basic-info-service
1       ts-admin-travel-service
2            ts-assurance-mongo
3          ts-assurance-service
4                 ts-auth-mongo
..                          ...
40            ts-travel-service
41             ts-travel2-mongo
42           ts-travel2-service
43              ts-ui-dashboard
44                ts-user-mongo

[45 rows x 1 columns],           log_name
0  log.error_count
1    log.row_count,           log_name  count
0  log.error_count   1098
1    log.row_count   1098, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing metric_df (already loaded and timestamp parsed to UTC)
selected = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

filtered = metric_df[metric_df['cmdb_id'].isin(selected)].copy()

# Group and compute aggregates
result = (
    filtered.groupby(['cmdb_id', 'kpi_name'], sort=True)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p50=('value', lambda x: x.quantile(0.50)),
        p90=('value', lambda x: x.quantile(0.90)),
        p95=('value', lambda x: x.quantile(0.95))
    )
    .reset_index()
    .sort_values(['cmdb_id', 'kpi_name'])
    .head(50)
)

result

```
Out[2]:
```
# Concise plain-English summary of the computed aggregates
summary = (
    "Summary of metric aggregates for selected services (top 50 rows shown):\n\n"
    "- Total groups returned: 37 (rows for each cmdb_id,kpi_name).\n"
    "- Most KPI series have 24–25 data points each.\n"
    "- Timestamp coverage: min timestamps mostly start at 2024-01-23 12:02:00+00:00 "
    "(a few start at 12:03:00+00:00); max timestamps go up to 2024-01-23 12:26:00+00:00.\n\n"
    "Examples (cmdb_id, kpi_name -> count, min_timestamp, max_timestamp, p50, p90, p95):\n"
    "- ts-auth-service, cpu -> 25, 2024-01-23 12:02 -> 2024-01-23 12:26, p50=14.91, p90=16.51, p95=17.11\n"
    "- ts-auth-service, diskio -> 25, same time window, p50=63052.46, p90=134641.0, p95=142349.6\n"
    "- ts-travel-service, mem -> 25, 2024-01-23 12:02 -> 2024-01-23 12:26, p50=4.93e8, p90=5.24e8, p95=5.40e8\n\n"
    "The full compact table returned contains per-group: [cmdb_id, kpi_name, count, min_timestamp, "
    "max_timestamp, p50, p90, p95] for the five services (sorted by cmdb_id then kpi_name)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count             min_timestamp             max_timestamp           p50           p90           p95
0     ts-auth-service         cpu     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  1.491142e+01  1.650792e+01  1.711169e+01
1     ts-auth-service      diskio     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  6.305246e+04  1.346410e+05  1.423496e+05
2     ts-auth-service  latency-50     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  2.447255e-01  2.862879e-01  2.937110e-01
3     ts-auth-service  latency-90     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  7.214000e-01  1.024790e+00  1.337515e+00
4     ts-auth-service         mem     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  2.409499e+08  2.412647e+08  2.414119e+08
..                ...         ...    ...                       ...                       ...           ...           ...           ...
32  ts-travel-service  latency-50     24 2024-01-23 12:03:00+00:00 2024-01-23 12:26:00+00:00  3.829360e-02  6.611768e-02  1.027198e-01
33  ts-travel-service  latency-90     24 2024-01-23 12:03:00+00:00 2024-01-23 12:26:00+00:00  3.152916e-01  5.032051e-01  8.787402e-01
34  ts-travel-service         mem     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  4.932643e+08  5.242371e+08  5.403971e+08
35  ts-travel-service      socket     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  1.840000e+01  2.208667e+01  2.708667e+01
36  ts-travel-service    workload     25 2024-01-23 12:02:00+00:00 2024-01-23 12:26:00+00:00  8.297783e+00  1.082803e+01  1.109505e+01

[37 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing variables: metric_df, result, selected (defined earlier)
# Define incident window in UTC
window_start = pd.to_datetime("2024-01-23 11:59:43", utc=True)
window_end = pd.to_datetime("2024-01-23 12:29:43", utc=True)

# Filter metric rows for selected services within the incident window
windowed = metric_df[
    metric_df['cmdb_id'].isin(selected) &
    (metric_df['timestamp'] >= window_start) &
    (metric_df['timestamp'] <= window_end)
].copy()

# Global p95 thresholds computed from full series (from 'result' variable)
global_p95 = result[['cmdb_id', 'kpi_name', 'p95']].copy()

# Merge global p95 into the windowed rows
windowed = windowed.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value > global p95 (NaN p95 will yield False)
windowed['is_anomaly'] = windowed['value'] > windowed['p95']

# Aggregates within the window
agg_total_max = (
    windowed.groupby(['cmdb_id', 'kpi_name'], sort=True)
    .agg(total_points=('value', 'count'), max_value=('value', 'max'))
    .reset_index()
)

# Anomaly-specific aggregates: count and earliest anomaly timestamp
anomaly_agg = (
    windowed[windowed['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], sort=True)
    .agg(anomaly_count=('is_anomaly', 'sum'), earliest_anomaly_timestamp=('timestamp', 'min'))
    .reset_index()
)

# Combine aggregates and include p95
combined = (
    agg_total_max
    .merge(anomaly_agg, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill missing anomaly counts with 0; keep earliest_anomaly_timestamp as NaT when none
combined['anomaly_count'] = combined['anomaly_count'].fillna(0).astype(int)

# Select and order columns as requested
combined = combined[['cmdb_id', 'kpi_name', 'total_points', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value', 'p95']]

# Sort by anomaly_count desc then max_value desc and limit to top 20
combined_sorted = combined.sort_values(by=['anomaly_count', 'max_value'], ascending=[False, False]).head(20).reset_index(drop=True)

# If no anomalies detected across all groups, return a message string per instructions
if combined_sorted['anomaly_count'].sum() == 0:
    output = "No anomalies detected in the incident window for the selected services."
else:
    output = combined_sorted

output

```
Out[3]:
```
summary = (
    "Summary of anomalies in the incident window (based on the returned top-20 rows):\n\n"
    "- Detected groups: 20 (these top 20 groups each had anomaly_count=2), totaling 40 anomaly points in the returned table.\n"
    "- Earliest anomaly (returned rows): 2024-01-23 12:02:00+00:00 UTC. Latest anomaly (returned rows): 2024-01-23 12:25:00+00:00 UTC.\n"
    "- Anomalous KPI types (frequent in the table): mem, diskio, cpu, socket, workload.\n"
    "- Highest-severity examples from the returned rows:\n"
    "  * ts-travel-service, diskio -> max=1.502491e+10, p95=1.496009e+10 (anomaly)\n"
    "  * ts-travel-service, mem   -> max=5.552704e+08, p95=5.403971e+08 (anomaly)\n"
    "  * ts-order-service, mem    -> max=2.661726e+08, p95=2.661073e+08 (anomaly)\n\n"
    "Conclusion: Multiple services show metric values exceeding their global p95 thresholds during the incident window, "
    "notably memory and disk I/O across several services (e.g., ts-travel-service, ts-order-service). Recommend inspecting those services' resource usage and related logs/traces."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name  total_points  anomaly_count earliest_anomaly_timestamp     max_value           p95
0   ts-travel-service    diskio            25              2  2024-01-23 12:20:00+00:00  1.502491e+10  1.496009e+10
1   ts-travel-service       mem            25              2  2024-01-23 12:04:00+00:00  5.552704e+08  5.403971e+08
2    ts-order-service       mem            25              2  2024-01-23 12:25:00+00:00  2.661726e+08  2.661073e+08
3    ts-train-service       mem            25              2  2024-01-23 12:25:00+00:00  2.562209e+08  2.561625e+08
4    ts-route-service       mem            25              2  2024-01-23 12:24:00+00:00  2.561071e+08  2.560942e+08
5     ts-auth-service       mem            25              2  2024-01-23 12:10:00+00:00  2.414699e+08  2.414119e+08
6    ts-order-service    diskio            25              2  2024-01-23 12:25:00+00:00  2.207535e+05  2.093363e+05
7     ts-auth-service    diskio            25              2  2024-01-23 12:16:00+00:00  1.496269e+05  1.423496e+05
8    ts-route-service    diskio            25              2  2024-01-23 12:05:00+00:00  6.476950e+04  6.015456e+04
9    ts-train-service    diskio            25              2  2024-01-23 12:11:00+00:00  5.905132e+04  5.760140e+04
10  ts-travel-service       cpu            25              2  2024-01-23 12:02:00+00:00  3.989239e+01  3.738659e+01
11   ts-route-service  workload            25              2  2024-01-23 12:12:00+00:00  3.717437e+01  3.650768e+01
12  ts-travel-service    socket            25              2  2024-01-23 12:04:00+00:00  2.973333e+01  2.708667e+01
13   ts-train-service  workload            25              2  2024-01-23 12:12:00+00:00  2.355177e+01  2.261267e+01
14    ts-auth-service       cpu            25              2  2024-01-23 12:05:00+00:00  1.825859e+01  1.711169e+01
15   ts-train-service    socket            25              2  2024-01-23 12:07:00+00:00  1.800000e+01  1.770333e+01
16    ts-auth-service    socket            25              2  2024-01-23 12:02:00+00:00  1.460000e+01  1.398000e+01
17   ts-route-service    socket            25              2  2024-01-23 12:05:00+00:00  1.385000e+01  1.365333e+01
18   ts-order-service    socket            25              2  2024-01-23 12:20:00+00:00  1.315000e+01  1.291485e+01
19  ts-travel-service  workload            25              2  2024-01-23 12:10:00+00:00  1.135227e+01  1.109505e+01```
```

In [ ]:
# In[4]:

# Reuse existing variables: trace_df, window_start, window_end, selected
import pandas as pd

# Service identifiers to match in trace_name
service_ids = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Filter trace rows where trace_name contains any of the service identifiers
mask = trace_df['trace_name'].astype(str).apply(lambda s: any(sid in s for sid in service_ids))
trace_matches = trace_df[mask].copy()

# 1) Compute global-series aggregates (using full available trace_matches series) BEFORE any window filtering
global_agg = (
    trace_matches.groupby('trace_name', sort=True)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.50),
         p90=lambda x: x.quantile(0.90),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .reset_index()
)

# 2) Filter trace_matches to the incident window
windowed_trace = trace_matches[
    (trace_matches['timestamp'] >= window_start) &
    (trace_matches['timestamp'] <= window_end)
].copy()

# Merge global p95 into windowed rows to evaluate anomalies
windowed_trace = windowed_trace.merge(global_agg[['trace_name', 'p95']], on='trace_name', how='left')

# Mark anomalies where value > global p95
windowed_trace['is_anomaly'] = windowed_trace['value'] > windowed_trace['p95']

# 3) Compute windowed aggregates per trace_name
window_agg = (
    windowed_trace.groupby('trace_name', sort=True)
    .agg(
        total_points_in_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x.min() if x.any() else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Note: groups that have no rows in window won't appear in window_agg; include them with zeros/NaT
combined = (
    global_agg
    .merge(window_agg, on='trace_name', how='left')
)

# Fill missing window aggregates with defaults: 0,0,NaT,NaN
combined['total_points_in_window'] = combined['total_points_in_window'].fillna(0).astype(int)
combined['anomaly_count'] = combined['anomaly_count'].fillna(0).astype(int)
combined['earliest_anomaly_timestamp'] = pd.to_datetime(combined['earliest_anomaly_timestamp'], utc=True)
combined['max_value_in_window'] = combined['max_value_in_window'].where(combined['max_value_in_window'].notna(), None)

# Reorder/rename columns to match requested output
trace_summary = combined.rename(columns={
    'count': 'count',
    'p50': 'p50',
    'p90': 'p90',
    'p95': 'p95',
    'max_value': 'max_value',
    'total_points_in_window': 'total_points_in_window',
    'anomaly_count': 'anomaly_count',
    'earliest_anomaly_timestamp': 'earliest_anomaly_timestamp',
    'max_value_in_window': 'max_value_in_window'
})[
    ['trace_name', 'count', 'p50', 'p90', 'p95', 'max_value', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
]

# Sort by anomaly_count desc then max_value_in_window desc, limit to top 20
trace_summary_sorted = trace_summary.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

trace_summary_sorted

```
Out[4]:
```
Summary of trace anomalies (plain English):

- Method: anomalies = values > global p95 (computed from full trace series). Window = 2024-01-23 11:59:43 to 2024-01-23 12:29:43 UTC.

- Highest anomaly groups:
  - trace.to_ts-order-service.row_count: count=171, p95=439, max=606, total_points_in_window=171, anomaly_count=9, earliest anomaly=2024-01-23 12:02:00 UTC, max_in_window=606.
  - trace.to_ts-order-service.duration_p95 and duration_mean: same counts (171) and anomaly_count=9, earliest anomaly=2024-01-23 12:02:00 UTC (max values ~0.6546s and ~0.5842s).
- Travel-related traces also show notable anomalies:
  - trace.to_ts-travel-service.duration_p95 / duration_mean: count=143, anomaly_count=8, earliest anomaly=2024-01-23 12:03:00 UTC (max duration ~3.20s / ~2.06s).
  - trace.from_ts-travel-service.* and row_count/to_ts-travel-service.row_count: counts 143–144, anomaly_count 7–8, earliest anomaly=2024-01-23 12:03:00 UTC (row_count max ~1952).
- Other services with repeated anomalies:
  - to_ts-train-service.row_count and to_ts-route-service.row_count: anomaly_count=5, earliest anomaly=2024-01-23 12:02:00 UTC (very large row_count spikes).
  - to_ts-auth-service.row_count and duration_p95: anomaly_count=3, earliest anomaly=2024-01-23 12:02:00 UTC.
- Time range: earliest detected anomalies appear at ~12:02–12:03 UTC; anomalies persist through the window up to ~12:25 UTC in some metrics.
- Interpretation / next steps: several upstream/downstream traces show spikes in row_count and increased durations (order-service, travel-service, train/route). This pattern suggests traffic surges and/or downstream processing bottlenecks affecting latency and throughput. Recommend inspecting request rates, service logs, and resource metrics (CPU/memory/disk I/O) for the implicated services (order, travel, train, route, auth).

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  count         p50          p90          p95    max_value  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0          trace.to_ts-order-service.row_count    171    9.000000   378.000000   439.000000   606.000000                     171              9  2024-01-23 12:02:00+00:00           606.000000
1       trace.to_ts-order-service.duration_p95    171    0.012808     0.526783     0.574277     0.654581                     171              9  2024-01-23 12:02:00+00:00             0.654581
2      trace.to_ts-order-service.duration_mean    171    0.010266     0.476099     0.533468     0.584241                     171              9  2024-01-23 12:02:00+00:00             0.584241
3      trace.to_ts-travel-service.duration_p95    143    0.162060     2.193389     2.614973     3.201895                     143              8  2024-01-23 12:03:00+00:00             3.201895
4     trace.to_ts-travel-service.duration_mean    143    0.067425     0.934884     1.278710     2.056019                     143              8  2024-01-23 12:03:00+00:00             2.056019
5    trace.from_ts-travel-service.duration_p95    144    0.031186     0.201761     0.217434     0.410321                     144              8  2024-01-23 12:03:00+00:00             0.410321
6   trace.from_ts-travel-service.duration_mean    144    0.009273     0.077906     0.117916     0.262489                     144              8  2024-01-23 12:03:00+00:00             0.262489
7       trace.from_ts-travel-service.row_count    144  218.500000  1370.800000  1517.000000  1952.000000                     144              7  2024-01-23 12:03:00+00:00          1952.000000
8         trace.to_ts-travel-service.row_count    143   20.000000  1373.200000  1517.000000  1952.000000                     143              7  2024-01-23 12:03:00+00:00          1952.000000
9          trace.to_ts-train-service.row_count     99  248.000000  2456.800000  2658.800000  2840.000000                      99              5  2024-01-23 12:02:00+00:00          2840.000000
10         trace.to_ts-route-service.row_count     99  361.000000  1978.400000  2098.400000  2248.000000                      99              5  2024-01-23 12:02:00+00:00          2248.000000
11      trace.to_ts-route-service.duration_p95     99    0.003703     0.003908     0.003996     0.007376                      99              5  2024-01-23 12:02:00+00:00             0.007376
12      trace.to_ts-train-service.duration_p95     99    0.004766     0.005114     0.005252     0.006496                      99              5  2024-01-23 12:02:00+00:00             0.006496
13     trace.to_ts-train-service.duration_mean     99    0.004021     0.004234     0.004323     0.004429                      99              5  2024-01-23 12:02:00+00:00             0.004429
14     trace.to_ts-route-service.duration_mean     99    0.003085     0.003247     0.003308     0.003508                      99              5  2024-01-23 12:02:00+00:00             0.003508
15       trace.from_ts-order-service.row_count     50   19.500000   481.200000   532.600000   606.000000                      50              3  2024-01-23 12:02:00+00:00           606.000000
16          trace.to_ts-auth-service.row_count     50  102.000000   342.200000   346.200000   352.000000                      50              3  2024-01-23 12:02:00+00:00           352.000000
17       trace.to_ts-auth-service.duration_p95     50    0.240630     0.896085     1.023678     1.609866                      50              3  2024-01-23 12:02:00+00:00             1.609866
18    trace.from_ts-order-service.duration_p95     50    0.423691     0.569757     0.598577     0.632175                      50              3  2024-01-23 12:02:00+00:00             0.632175
19   trace.from_ts-order-service.duration_mean     50    0.234301     0.524125     0.532653     0.564407                      50              3  2024-01-23 12:02:00+00:00             0.564407```
```

In [ ]:
# In[5]:

# Reuse existing variables: log_df, selected, window_start, window_end
import pandas as pd

# 1) Filter logs to the selected services (global series computation must be before window filtering)
filtered_logs = log_df[log_df['cmdb_id'].isin(selected)].copy()

# Compute global statistics per (cmdb_id, log_name)
global_log_agg = (
    filtered_logs.groupby(['cmdb_id', 'log_name'], sort=True)['value']
    .agg(
        count='count',
        p50=lambda x: x.quantile(0.50),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

# 2) Restrict to the incident window
windowed_logs = filtered_logs[
    (filtered_logs['timestamp'] >= window_start) &
    (filtered_logs['timestamp'] <= window_end)
].copy()

# Merge global p95 to evaluate anomalies
windowed_logs = windowed_logs.merge(global_log_agg[['cmdb_id', 'log_name', 'p95']], on=['cmdb_id', 'log_name'], how='left')

# Mark anomalies where value > global p95
windowed_logs['is_anomaly'] = windowed_logs['value'] > windowed_logs['p95']

# 3) Windowed aggregates: total points and max value in window
window_totals = (
    windowed_logs.groupby(['cmdb_id', 'log_name'], sort=True)
    .agg(
        total_points_in_window=('value', 'count'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 4) Anomaly-specific aggregates in window
anomaly_stats = (
    windowed_logs[windowed_logs['is_anomaly']]
    .groupby(['cmdb_id', 'log_name'], sort=True)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min')
    )
    .reset_index()
)

# 5) Combine global and windowed results
combined = (
    global_log_agg
    .merge(window_totals, on=['cmdb_id', 'log_name'], how='left')
    .merge(anomaly_stats, on=['cmdb_id', 'log_name'], how='left')
)

# Fill defaults for groups without window rows / anomalies
combined['total_points_in_window'] = combined['total_points_in_window'].fillna(0).astype(int)
combined['anomaly_count'] = combined['anomaly_count'].fillna(0).astype(int)
combined['earliest_anomaly_timestamp'] = pd.to_datetime(combined['earliest_anomaly_timestamp'], utc=True)
combined['max_value_in_window'] = combined['max_value_in_window'].where(combined['max_value_in_window'].notna(), None)

# Select and order columns as requested
log_summary = combined[
    ['cmdb_id', 'log_name', 'count', 'p50', 'p90', 'p95', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
].copy()

# Sort by anomaly_count desc then max_value_in_window desc, limit to top 20
log_summary_sorted = log_summary.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# If no anomalies detected, return a clear message string; otherwise return the compact table
if log_summary_sorted['anomaly_count'].sum() == 0:
    output = "No anomalies detected in logs for the selected services within the incident window."
else:
    output = log_summary_sorted

output

```
Out[5]:
```
summary = (
    "Log analysis (window 2024-01-23 11:59:43 to 2024-01-23 12:29:43 UTC):\n\n"
    "- Elevated log.row_count detected for four services (each had 25 points in-window and 2 anomaly points where value > global p95):\n"
    "  * ts-travel-service, log.row_count — global p95=2052.2, anomaly_count=2, earliest anomaly=2024-01-23 12:12:00 UTC, max_in_window=2174\n"
    "  * ts-route-service, log.row_count  — global p95=1082.6, anomaly_count=2, earliest anomaly=2024-01-23 12:08:00 UTC, max_in_window=1124\n"
    "  * ts-order-service, log.row_count  — global p95=472.2, anomaly_count=2, earliest anomaly=2024-01-23 12:12:00 UTC, max_in_window=512\n"
    "  * ts-auth-service, log.row_count   — global p95=349.6, anomaly_count=2, earliest anomaly=2024-01-23 12:08:00 UTC, max_in_window=352\n\n"
    "- log.error_count for these services shows no anomalies (p95=0 and anomaly_count=0).\n\n"
    "Interpretation: Several services experienced short spikes in request/row counts during the incident window (not increases in error counts). Investigate traffic patterns, upstream callers, and resource usage (CPU/memory/disk I/O) on the implicated services (travel, route, order, auth)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  count     p50     p90     p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0  ts-travel-service    log.row_count     25  1572.0  2018.2  2052.2                      25              2  2024-01-23 12:12:00+00:00                 2174
1   ts-route-service    log.row_count     25   988.0  1081.0  1082.6                      25              2  2024-01-23 12:08:00+00:00                 1124
2   ts-order-service    log.row_count     25   352.0   451.8   472.2                      25              2  2024-01-23 12:12:00+00:00                  512
3    ts-auth-service    log.row_count     25   325.0   346.4   349.6                      25              2  2024-01-23 12:08:00+00:00                  352
4    ts-auth-service  log.error_count     25     0.0     0.0     0.0                      25              0                        NaT                    0
5   ts-order-service  log.error_count     25     0.0     0.0     0.0                      25              0                        NaT                    0
6   ts-route-service  log.error_count     25     0.0     0.0     0.0                      25              0                        NaT                    0
7  ts-travel-service  log.error_count     25     0.0     0.0     0.0                      25              0                        NaT                    0```
```

In [ ]:
# In[6]:

# Reuse existing variables: trace_df, window_start, window_end
import re
import pandas as pd

service_ids = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# 1) Select trace rows that mention any candidate service
mask = trace_df['trace_name'].astype(str).apply(lambda s: any(sid in s for sid in service_ids))
trace_matches = trace_df[mask].copy()

# 2) Compute global p95 (and other globals) per trace_name using full series
global_trace_agg = (
    trace_matches.groupby('trace_name', sort=True)['value']
    .agg(count='count',
         p50=lambda x: x.quantile(0.50),
         p90=lambda x: x.quantile(0.90),
         p95=lambda x: x.quantile(0.95),
         max_value='max')
    .reset_index()
)

# 3) Windowed rows and per-trace_name anomaly detection using the precomputed global p95
windowed_trace = trace_matches[
    (trace_matches['timestamp'] >= window_start) &
    (trace_matches['timestamp'] <= window_end)
].copy()

# Merge global p95 into windowed data
windowed_trace = windowed_trace.merge(global_trace_agg[['trace_name', 'p95']], on='trace_name', how='left')

# Mark anomalies where value > global p95
windowed_trace['is_anomaly'] = windowed_trace['value'] > windowed_trace['p95']

# Aggregate per trace_name within window
trace_window_agg = (
    windowed_trace.groupby('trace_name', sort=True)
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x.min() if x.any() else pd.NaT),
        max_value_in_window=('value', 'max'),
        total_points_in_window=('value', 'count')
    )
    .reset_index()
)

# Ensure trace_names with no rows in window appear with zeros by merging with global_trace_agg
trace_combined = global_trace_agg[['trace_name', 'p95']].merge(trace_window_agg, on='trace_name', how='left')
trace_combined['anomaly_count'] = trace_combined['anomaly_count'].fillna(0).astype(int)
trace_combined['earliest_anomaly_timestamp'] = pd.to_datetime(trace_combined['earliest_anomaly_timestamp'], utc=True)
trace_combined['max_value_in_window'] = trace_combined['max_value_in_window'].where(trace_combined['max_value_in_window'].notna(), None)
trace_combined['total_points_in_window'] = trace_combined['total_points_in_window'].fillna(0).astype(int)

# 4) Map each trace_name to a service by finding the first matching service_id in the trace_name
def extract_service(trace_name):
    for sid in service_ids:
        if sid in trace_name:
            return sid
    return None

trace_combined['service'] = trace_combined['trace_name'].apply(extract_service)

# Drop any trace_names that didn't map (shouldn't happen given initial filter)
trace_combined = trace_combined[trace_combined['service'].notna()].copy()

# Aggregate per service: sum anomaly counts, earliest anomaly timestamp (min), and max of max_value_in_window
service_agg = (
    trace_combined.groupby('service', sort=True)
    .agg(
        total_trace_anomaly_count=('anomaly_count', 'sum'),
        earliest_anomaly_timestamp=('earliest_anomaly_timestamp', lambda x: x.min() if x.notna().any() else pd.NaT),
        max_value_in_window_across_traces=('max_value_in_window', lambda x: pd.Series(x).dropna().astype(float).max() if pd.Series(x).notna().any() else None)
    )
    .reset_index()
)

# Keep only the five candidate services and sort by anomaly count desc
service_summary = service_agg[service_agg['service'].isin(service_ids)].sort_values(
    by=['total_trace_anomaly_count', 'max_value_in_window_across_traces'],
    ascending=[False, False]
).reset_index(drop=True)

service_summary

```
Out[6]:
```
summary = (
    "Trace-based service anomaly summary (incident window 2024-01-23 11:59:43–12:29:43 UTC):\n\n"
    "- ts-travel-service: total_trace_anomaly_count=46; earliest anomaly=2024-01-23 12:03:00+00:00; max value in-window=1952.0\n"
    "- ts-order-service:  total_trace_anomaly_count=36; earliest anomaly=2024-01-23 12:02:00+00:00; max value in-window=606.0\n"
    "- ts-train-service:  total_trace_anomaly_count=21; earliest anomaly=2024-01-23 12:02:00+00:00; max value in-window=2840.0\n"
    "- ts-route-service:  total_trace_anomaly_count=21; earliest anomaly=2024-01-23 12:02:00+00:00; max value in-window=2248.0\n"
    "- ts-auth-service:   total_trace_anomaly_count=15; earliest anomaly=2024-01-23 12:02:00+00:00; max value in-window=352.0\n\n"
    "Interpretation: ts-travel-service shows the most trace anomalies (46) with the earliest detected spike at ~12:03 UTC and large row_count spikes (max ~1952). "
    "Order, train, route, and auth also show significant anomalies (earliest ~12:02 UTC). These results point to widespread request/processing spikes affecting multiple services, with ts-travel-service the most affected by trace anomalies."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

service  total_trace_anomaly_count earliest_anomaly_timestamp  max_value_in_window_across_traces
0  ts-travel-service                         46  2024-01-23 12:03:00+00:00                             1952.0
1   ts-order-service                         36  2024-01-23 12:02:00+00:00                              606.0
2   ts-train-service                         21  2024-01-23 12:02:00+00:00                             2840.0
3   ts-route-service                         21  2024-01-23 12:02:00+00:00                             2248.0
4    ts-auth-service                         15  2024-01-23 12:02:00+00:00                              352.0```
```

In [ ]:
# In[7]:

# Reuse trace_df already loaded
import pandas as pd

service_ids = ['ts-order-service', 'ts-travel-service', 'ts-train-service', 'ts-route-service', 'ts-auth-service']

# Count how many candidate service identifiers appear in each trace_name
trace_name_series = trace_df['trace_name'].astype(str)
matches_count = trace_name_series.apply(lambda s: sum(1 for sid in service_ids if sid in s))

# Select trace_names that mention at least two different candidate services
pair_traces = trace_df[matches_count >= 2].copy()

# Group by trace_name and count occurrences, sort descending, limit to top 50
pair_trace_counts = (
    pair_traces.groupby('trace_name', sort=True)
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

pair_trace_counts

```
Out[7]:
```
No trace_name contains two different candidate services together. The search returned zero matches (no trace_name with pairs of the services ts-order-service, ts-travel-service, ts-train-service, ts-route-service, ts-auth-service). 

Implication: there are no trace_name strings that directly show call relationships between these candidate services, so downstream ordering cannot be inferred from trace_name contents alone.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [trace_name, count]
Index: []```
```